# Use Case Test

## Solution

### GNNs

1. Modellierung der "Topologischen Realität"

    - Herkömmliche Modelle rechnen mit Luftlinien (Euklidische Distanz). Ein GNN arbeitet auf der tatsächlichen Graph-Struktur (Straßen, Wege, Barrieren).
    - Vorteil: Das GNN versteht, dass ein Park, der zwar nah ist, aber durch eine Bahnlinie getrennt wird, für den User "weit weg" ist. Es lernt die Konnektivität, nicht nur die Position.

2. Relationales Lernen (Der "Synergie-Effekt")

    - In der Immobilienbewertung oder Standortanalyse ist ein POI nie isoliert zu betrachten.
    - Vorteil: Ein GNN nutzt Message Passing. Ein Knoten (z. B. eine U-Bahn-Station) "sendet" seinen Wert an benachbarte Knoten (Cafés, Wohnungen). Das Modell lernt automatisch, dass ein Café neben einer Haltestelle wertvoller ist als ein isoliertes Café. Diese Abhängigkeiten (Interdependencies) kann kein normales Regressionsmodell erfassen.

3. Feature-Aggregation über Nachbarschaften (Multi-Hop)

    - Ein GNN kann Informationen über mehrere "Hops" hinweg aggregieren.
    - Vorteil: Du kannst berechnen, wie die Qualität eines Viertels (2. oder 3. Grad der Nachbarschaft) auf deine Zieladresse abstrahlt. Das Modell "sieht" das gesamte lokale Ökosystem und gewichtet automatisch, welcher Einflussfaktor (z. B. Kriminalität im Nachbarviertel vs. Park vor der Tür) schwerer wiegt.

4. Umgang mit unstrukturierten & unvollständigen Daten

    - OSM-Daten sind oft unvollständig (wie wir beim Solar-Beispiel gesehen haben).
    - Vorteil: GNNs sind hervorragend darin, fehlende Knoten-Attribute zu inferieren (Node Classification / Link Prediction). Wenn alle Häuser in der Umgebung "Typ A" sind, kann das GNN mit hoher Wahrscheinlichkeit vorhersagen, welche Eigenschaften ein unmarkiertes Haus hat.

5. Invarianz gegenüber der Anzahl der POIs

    - Ein klassisches Neural Network benötigt einen festen Input-Vektor (z. B. immer genau 10 Features).
    - Vorteil: GNNs sind input-agnostisch. Egal ob an einer Adresse 5 oder 500 POIs im Radius liegen – der Graph-Faltungsschicht (Graph Convolution) ist das egal. Sie verarbeitet die Struktur so, wie sie vorhanden ist.

**Zusammenfassung für dein Projekt-Pitch:**

- "Ich nutze PyTorch GNNs, weil urbane Räume keine Tabellen sind, sondern Graphen. Während klassische Scoring-Modelle POIs nur starr addieren, lernt mein GNN die impliziten Beziehungen und Synergien zwischen Standorten. Es versteht nicht nur was da ist, sondern wie die Anordnung der Stadtteile den Wert einer Adresse dynamisch beeinflusst."

## Possible POIs

1. Versorgung & Alltag (amenity & shop)

    Dies sind die "Brot und Butter" Features für jedes Scoring-Modell.

    - Lebensmittel: shop=supermarket, shop=bakery, shop=convenience
    - Gastronomie: amenity=cafe, amenity=restaurant, amenity=fast_food, amenity=pub
    - Gesundheit: amenity=pharmacy, amenity=doctors, amenity=hospital
    - Banken: amenity=bank, amenity=atm

2. Mobilität & Erreichbarkeit (highway & amenity)

    Für ein GNN sind diese besonders wichtig, da sie die "Knotenpunkte" des Netzwerks darstellen.

    - ÖPNV: highway=bus_stop, railway=station, railway=tram_stop
    - Individualverkehr: amenity=parking, amenity=charging_station, amenity=fuel
    - Infrastruktur: highway=cycleway, highway=pedestrian (Fußgängerzonen)

3. Freizeit & Lebensqualität (leisure & tourism)

    Diese Faktoren "strahlen" oft positiv auf die Umgebung ab (Value Boost).

    - Grünflächen: leisure=park, leisure=playground, leisure=garden
    - Sport: leisure=sports_centre, leisure=swimming_pool, leisure=fitness_centre
    - Kultur: tourism=museum, amenity=cinema, amenity=theatre

4. Bildung & Soziales (amenity)

    Besonders relevant für Familien-Scoring oder demografische Analysen.

    - Bildung: amenity=kindergarten, amenity=school, amenity=university, amenity=library
    - Öffentlich: amenity=townhall, amenity=post_office, amenity=place_of_worship

## Define Quality Metrics

In [ ]:
# TODO: Adjust metrics to personal preferences
QUALITY_METRICS = {
    "daily_needs": {
        "description": "Access to essential services like supermarkets and pharmacies",
        "weight": 0.40,
    },
    "leisure": {
        "description": "Access to parks, sports, and recreation",
        "weight": 0.25,
    },
    "convenience": {
        "description": "How close the nearest important services are",
        "weight": 0.20,
    },
    "diversity": {
        "description": "Variety of useful POI categories nearby",
        "weight": 0.15,
    },
}

## Define Measurable Metrics

In [ ]:
# TODO: Adjust metrics to personal preferences
MEASURABLE_METRICS = {
    "daily_needs": [
        {"feature": "supermarket_count_1000m", "type": "count", "weight": 0.6},
        {"feature": "pharmacy_count_1000m", "type": "count", "weight": 0.4},
    ],
    "leisure": [
        {"feature": "park_count_1000m", "type": "count", "weight": 0.5},
        {"feature": "sports_centre_count_2000m", "type": "count", "weight": 0.3},
        {"feature": "swimming_pool_count_2000m", "type": "count", "weight": 0.2},
    ],
    "convenience": [
        {"feature": "nearest_supermarket_m", "type": "distance", "weight": 0.4},
        {"feature": "nearest_pharmacy_m", "type": "distance", "weight": 0.3},
        {"feature": "nearest_park_m", "type": "distance", "weight": 0.3},
    ],
    "diversity": [
        {"feature": "poi_diversity_1000m", "type": "count", "weight": 1.0},
    ],
}

## Application

In [ ]:
import os
import json
import requests
import random
from typing import Iterable, List, Dict, Any, Optional, Tuple

import pandas as pd
import numpy as np
import folium

from geopy.geocoders import Nominatim
from geopy.distance import geodesic

### Helper functions

In [ ]:
def get_user_location(address_or_coords: str) -> Optional[Tuple[float, float]]:
    """Resolve a human-readable address to (lat, lon)."""
    geolocator = Nominatim(user_agent="smls_gnn_project")

    try:
        location = geolocator.geocode(address_or_coords, exactly_one=True, timeout=20)
        if location is None:
            print("Address could not be found.")
            return None

        print("Location confirmed")
        return (location.latitude, location.longitude)

    except Exception as e:
        print(f"Error during geocoding: {e}")
        return None

In [ ]:
def build_overpass_query(
    lat: float,
    lon: float,
    categories: Iterable[str],
    radius: int,
) -> str:
    """
    Build an Overpass QL query for multiple tag filters around a point.
    categories must look like: ['amenity=cafe', 'shop=supermarket']
    """
    parts = []

    for cat in categories:
        if "=" not in cat:
            raise ValueError(f"Invalid category format: {cat!r}. Expected 'key=value'.")

        key, value = cat.split("=", 1)
        key = key.strip()
        value = value.strip()

        # nwr = nodes, ways, relations
        parts.append(f'nwr["{key}"="{value}"](around:{radius},{lat},{lon});')

    query = f"""
    [out:json][timeout:90];
    (
      {"".join(parts)}
    );
    out center tags;
    """
    return query.strip()

In [ ]:
def fetch_poi_data(
    lat: float,
    lon: float,
    categories: Iterable[str],
    radius: int,
    overpass_url: Optional[str] = None,
    session: Optional[requests.Session] = None,
) -> List[Dict[str, Any]]:
    """Fetch POI data from Overpass. Always returns a list."""
    overpass_url = overpass_url or DEFAULT_OVERPASS_URL

    if not overpass_url:
        raise ValueError("Overpass URL is not provided.")

    query = build_overpass_query(lat, lon, categories, radius)
    http = session or requests.Session()

    try:
        response = http.post(
            overpass_url,
            data=query,
            headers={"Content-Type": "text/plain; charset=utf-8"},
            timeout=120,
        )

        # Raise on HTTP errors first
        response.raise_for_status()

        # Parse JSON explicitly and show a useful snippet if it fails
        try:
            payload = response.json()
        except json.JSONDecodeError:
            body_preview = response.text[:1000].strip()
            raise RuntimeError(
                "Overpass returned a non-JSON response.\n"
                f"URL: {overpass_url}\n"
                f"Status: {response.status_code}\n"
                f"Body preview:\n{body_preview}"
            )

        if "elements" not in payload:
            raise RuntimeError(
                "Overpass response JSON does not contain 'elements'. "
                f"Top-level keys: {list(payload.keys())}"
            )

        return payload["elements"]

    except requests.Timeout:
        raise RuntimeError("Overpass request timed out.")
    except requests.RequestException as e:
        raise RuntimeError(f"HTTP error during Overpass request: {e}") from e

In [ ]:
def visualize_gnn_nodes(center_lat, center_lon, processed_nodes, radius):
    """Visualizes the target address and POI nodes on a Folium map."""
    # Initialize the map centered on the target address
    m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles="cartodbpositron")

    # Draw the search radius as a circle around the target address
    folium.Circle(
        location=[center_lat, center_lon],
        radius=radius,
        color='blue',
        fill=True,
        fill_opacity=0.05,
        popup="Suchradius"
    ).add_to(m)

    # Mark the target address (Node 0) with a distinct marker
    folium.Marker(
        [center_lat, center_lon],
        popup="<b>ZIELADRESSE (Node 0)</b>",
        icon=folium.Icon(color='red', icon='home')
    ).add_to(m)

    # Mark the POI nodes with green circle markers
    for node in processed_nodes:
        if node['id'] == 'target_address':
            continue
            
        # Extract name and type from tags for the popup
        tags = node.get('tags', {})
        name = tags.get('name', 'Unbenannt')
        
        # Create a readable label from the tags
        category = next((f"{k}={v}" for k, v in tags.items() if k != 'name'), "POI")
        
        popup_text = f"<b>{name}</b><br>Kategorie: {category}<br>Distanz: {int(node['dist_to_origin'])}m"
        
        folium.CircleMarker(
            location=[node['lat'], node['lon']],
            radius=5,
            popup=popup_text,
            color='green',
            fill=True,
            fill_color='green',
            fill_opacity=0.7
        ).add_to(m)

    return m

### Get data for specified input

In [ ]:
# DEFAULT_OVERPASS_URL = "https://overpass.kumi.systems/api/interpreter"
DEFAULT_OVERPASS_URL = "https://overpass-api.de/api/interpreter"

In [ ]:
# User Input
address = os.getenv("ADDRESS")
points_of_interest = [
    'amenity=cafe', 
    'shop=supermarket', 
    'amenity=pharmacy', 
    'leisure=park',
    'leisure=swimming_pool',
    'leisure=sports_centre',
]
radius = 5000

if not address:
    raise ValueError("ADDRESS environment variable is missing.") 

In [ ]:
coords = get_user_location(address)
if coords is None:
    raise RuntimeError("Could not resolve address.")

lat, lon = coords
print(f'Success: Address resolved to coordinates: ({lat}, {lon})')

In [ ]:
raw_data = fetch_poi_data(
    lat=lat,
    lon=lon,
    categories=points_of_interest,
    radius=radius,
)

### Normalize raw OSM data

In [ ]:
def normalize_pois(raw_data):
    rows = []

    for el in raw_data:
        tags = el.get("tags", {})
        lat = el.get("lat")
        lon = el.get("lon")

        if lat is None or lon is None:
            center = el.get("center", {})
            lat = center.get("lat")
            lon = center.get("lon")

        category_group = None
        category_value = None
        for key in ["amenity", "shop", "leisure"]:
            if key in tags:
                category_group = key
                category_value = tags[key]
                break

        rows.append({
            "osm_id": el.get("id"),
            "osm_type": el.get("type"),
            "lat": lat,
            "lon": lon,
            "name": tags.get("name"),
            "category_group": category_group,
            "category_value": category_value,
            "tags": tags,
        })

    return pd.DataFrame(rows)

In [ ]:
poi_df = normalize_pois(raw_data)
poi_df

### Compute measurable metrics for one address

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000.0  # meters

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [ ]:
def compute_measurable_metrics(lat, lon, poi_df):
    df = poi_df.copy()

    df = df.dropna(subset=["lat", "lon", "category_value"]).copy()

    df["distance_m"] = haversine_distance(
        lat,
        lon,
        df["lat"].values,
        df["lon"].values
    )

    features = {}

    def count_within(category, radius):
        return int(((df["category_value"] == category) & (df["distance_m"] <= radius)).sum())

    def nearest_distance(category):
        cat_df = df[df["category_value"] == category]
        if cat_df.empty:
            return 99999.0
        return float(cat_df["distance_m"].min())

    # Daily needs
    features["supermarket_count_1000m"] = count_within("supermarket", 1000)
    features["pharmacy_count_1000m"] = count_within("pharmacy", 1000)

    # Leisure
    features["park_count_1000m"] = count_within("park", 1000)
    features["sports_centre_count_2000m"] = count_within("sports_centre", 2000)
    features["swimming_pool_count_2000m"] = count_within("swimming_pool", 2000)

    # Convenience
    features["nearest_supermarket_m"] = nearest_distance("supermarket")
    features["nearest_pharmacy_m"] = nearest_distance("pharmacy")
    features["nearest_park_m"] = nearest_distance("park")

    # Diversity
    features["poi_diversity_1000m"] = int(
        df.loc[df["distance_m"] <= 1000, "category_value"].nunique()
    )

    return features

In [ ]:
features = compute_measurable_metrics(lat, lon, poi_df)
features

In [ ]:
def normalize_count(value, cap):
    """
    Higher is better.
    Maps 0..cap to 0..1, clipping above cap.
    """
    return min(value / cap, 1.0)


def normalize_distance(value, best, worst):
    """
    Lower is better.
    <= best   -> 1.0
    >= worst  -> 0.0
    linear in between
    """
    if value <= best:
        return 1.0
    if value >= worst:
        return 0.0
    return 1.0 - (value - best) / (worst - best)

In [ ]:
def compute_quality_subscores(features):
    subscores = {}

    # daily_needs
    daily_needs = (
        0.6 * normalize_count(features["supermarket_count_1000m"], cap=3) +
        0.4 * normalize_count(features["pharmacy_count_1000m"], cap=2)
    )
    subscores["daily_needs"] = daily_needs

    # leisure
    leisure = (
        0.5 * normalize_count(features["park_count_1000m"], cap=3) +
        0.3 * normalize_count(features["sports_centre_count_2000m"], cap=2) +
        0.2 * normalize_count(features["swimming_pool_count_2000m"], cap=2)
    )
    subscores["leisure"] = leisure

    # convenience
    convenience = (
        0.4 * normalize_distance(features["nearest_supermarket_m"], best=100, worst=2000) +
        0.3 * normalize_distance(features["nearest_pharmacy_m"], best=100, worst=2000) +
        0.3 * normalize_distance(features["nearest_park_m"], best=150, worst=2500)
    )
    subscores["convenience"] = convenience

    # diversity
    diversity = normalize_count(features["poi_diversity_1000m"], cap=6)
    subscores["diversity"] = diversity

    return subscores

In [ ]:
def compute_final_score(subscores, quality_metrics=QUALITY_METRICS):
    score_0_1 = 0.0

    for metric_name, metric_info in quality_metrics.items():
        weight = metric_info["weight"]
        score_0_1 += weight * subscores[metric_name]

    return score_0_1 * 100.0

In [ ]:
features = compute_measurable_metrics(lat, lon, poi_df)
subscores = compute_quality_subscores(features)
final_score = compute_final_score(subscores)

print("Features:", features)
print("Subscores:", subscores)
print("Final score:", final_score)

In [ ]:
def score_address(lat, lon, poi_df):
    features = compute_measurable_metrics(lat, lon, poi_df)
    subscores = compute_quality_subscores(features)
    final_score = compute_final_score(subscores)

    return {
        "final_score": round(final_score, 2),
        "subscores": {k: round(v, 3) for k, v in subscores.items()},
        "features": features,
    }

In [ ]:
result = score_address(lat, lon, poi_df)
result

In [ ]:
addresses = [
    {"id": "A", "lat": 50.8396, "lon": 7.2090},
    {"id": "B", "lat": 50.8450, "lon": 7.2200},
    {"id": "C", "lat": 50.8500, "lon": 7.2000},
]

rows = []

for addr in addresses:
    result = score_address(addr["lat"], addr["lon"], poi_df)
    rows.append({
        "id": addr["id"],
        "lat": addr["lat"],
        "lon": addr["lon"],
        "final_score": result["final_score"],
        **result["subscores"],
    })

ranking_df = pd.DataFrame(rows).sort_values("final_score", ascending=False)
ranking_df